# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available Record Sets using their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata. Please check the dataset schema.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")

    # For each record set, print its Fields with their @id
    print("\nFields for each Record Set:")
    for rs in record_sets:
        fields = rs.get('field', [])
        # Ensure fields is a list
        if isinstance(fields, dict):
            fields = [fields]
        print(f"\nRecord Set: {rs['@id']}")
        for fld in fields:
            if isinstance(fld, dict):
                field_id = fld.get('@id','')
                field_name = fld.get('name', '')
                print(f"    Field @id: {field_id} | Name: {field_name}")
            else:
                print(f"    Field: {fld}")

    # Show sample records from the first record set
    print("\nSample records from the first Record Set:")
    record_set_id = record_sets[0]['@id']
    for idx, x in enumerate(dataset.records(record_set=record_set_id)):
        print(x)
        if idx > 2:
            break

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @id values
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
dataframes = {}
if not record_set_ids:
    print("No record sets available for extraction.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded DataFrame for record set {record_set_id} with {df.shape[0]} rows and columns: ")
            print(df.columns.tolist())
            display(df.head(3))
        else:
            print(f"No records found for record set {record_set_id}.")

# Pick the first non-empty dataframe for further analysis
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if not main_record_set_id:
    print("No valid DataFrame loaded from record sets.")
else:
    print(f"\nUsing Record Set @id for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

In [ ]:
# Identify numeric fields by their @id
import numpy as np

if main_record_set_id is not None:
    df_main = dataframes[main_record_set_id]
    # Attempt to infer numeric columns
    numeric_columns = df_main.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns: {numeric_columns}")
    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # Choose first numeric
        threshold = df_main[numeric_field_id].mean()
        filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / (filtered_df[numeric_field_id].std() + 1e-8)
        )
        print(f"\nNormalized field '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by the first non-numeric field
        non_numeric_columns = [col for col in df_main.columns if col not in numeric_columns]
        group_field = non_numeric_columns[0] if non_numeric_columns else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped average of '{numeric_field_id}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field for grouping.")
    else:
        print("No numeric fields found in the DataFrame.")
else:
    print("Cannot perform EDA: No valid DataFrame loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of numeric field distribution
if main_record_set_id is not None and numeric_columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped, barplot
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.barplot(
            data=grouped_df,
            x=group_field,
            y=numeric_field_id
        )
        plt.title(f"Mean of '{numeric_field_id}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded and inspected metadata and record sets from the Croissant dataset.
- Extracted main record set as a DataFrame and identified numeric fields for initial processing.
- Filtered and normalized numeric data, and (if possible) grouped by categorical attribute using `@id` naming conventions throughout.
- Visualized distributions of numeric data.
- This notebook provides a foundation for more detailed statistical analysis and predictive modeling using the Croissant FAIR^2 dataset.